# 数据治理 — Python 等效实现 (DataPilot)

本 notebook 与 `01_DataPilot.html` 功能完全对等，使用纯 Python 实现：
- CSV 加载与预览
- 4 种异常检测算法（Z-Score / IQR / Isolation Forest / MAD）
- 5 种缺失填补策略
- AI 特征工程
- 5 维数据质量评分

**对接案例数据集**：莱势明 50,000 工单数据


In [ ]:
# 环境准备
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.impute import KNNImputer
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

## 1. 数据加载与预览

In [ ]:
df = pd.read_csv("../datasets/work_orders.csv")
print(f"数据规模: {df.shape}")
print(f"字段: {df.columns.tolist()}")
df.head()

## 2. 异常检测 — Z-Score

In [ ]:
def detect_zscore(series, threshold=3):
    z = (series - series.mean()) / series.std()
    return np.abs(z) > threshold

df["proc_outlier_zscore"] = detect_zscore(df["estimated_proc_hours"], threshold=3)
print(f"Z-Score 检测异常数: {df['proc_outlier_zscore'].sum()} ({df['proc_outlier_zscore'].mean()*100:.2f}%)")

## 3. 异常检测 — IQR (四分位距)

In [ ]:
def detect_iqr(series, multiplier=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (series < q1 - multiplier * iqr) | (series > q3 + multiplier * iqr)

df["proc_outlier_iqr"] = detect_iqr(df["estimated_proc_hours"], multiplier=1.5)
print(f"IQR 检测异常数: {df['proc_outlier_iqr'].sum()} ({df['proc_outlier_iqr'].mean()*100:.2f}%)")

## 4. 异常检测 — Isolation Forest (孤立森林)

In [ ]:
# 多维度孤立森林
features = ["qty", "estimated_proc_hours"]
iso = IsolationForest(contamination=0.05, random_state=42)
df["outlier_iforest"] = iso.fit_predict(df[features]) == -1
print(f"Isolation Forest 检测异常: {df['outlier_iforest'].sum()}")

# 可视化
fig, ax = plt.subplots(figsize=(10, 6))
normal = df[~df["outlier_iforest"]]
outliers = df[df["outlier_iforest"]]
ax.scatter(normal["qty"], normal["estimated_proc_hours"], alpha=0.3, s=8, label="正常")
ax.scatter(outliers["qty"], outliers["estimated_proc_hours"], color="red", s=20, label="异常")
ax.set_xlabel("订单数量"); ax.set_ylabel("加工时长 (h)")
ax.set_title("Isolation Forest 异常检测"); ax.legend()
plt.show()

## 5. 缺失填补 — KNN

In [ ]:
# 模拟若干缺失
df_with_missing = df.copy()
mask = np.random.random(len(df_with_missing)) < 0.05
df_with_missing.loc[mask, "estimated_proc_hours"] = np.nan
print(f"缺失数: {df_with_missing['estimated_proc_hours'].isna().sum()}")

imputer = KNNImputer(n_neighbors=5)
df_with_missing["proc_filled_knn"] = imputer.fit_transform(
    df_with_missing[["qty", "estimated_proc_hours"]])[:, 1]
print(f"填补后缺失: {df_with_missing['proc_filled_knn'].isna().sum()}")
df_with_missing[mask][["order_id", "qty", "estimated_proc_hours", "proc_filled_knn"]].head()

## 6. AI 特征工程

In [ ]:
# 派生特征：紧急程度、是否周末等
df["due_time"] = pd.to_datetime(df["due_time"])
df["create_time"] = pd.to_datetime(df["create_time"])
df["urgency_hours"] = (df["due_time"] - df["create_time"]).dt.total_seconds() / 3600
df["is_weekend"] = df["create_time"].dt.weekday >= 5
df["estimated_value"] = df["qty"] * df["estimated_proc_hours"]  # 工单价值代理变量

print("派生特征已生成：")
print(df[["urgency_hours", "is_weekend", "estimated_value"]].describe())

## 7. 5 维数据质量评分

In [ ]:
def quality_score(df):
    n = len(df)
    completeness = 1 - df.isnull().sum().sum() / (n * df.shape[1])
    consistency = 1 - df.duplicated().sum() / n
    accuracy = (df["estimated_proc_hours"] > 0).sum() / n
    timeliness = ((pd.to_datetime("2026-04-30") - df["create_time"]).dt.days < 730).mean()
    uniqueness = df["order_id"].nunique() / n
    return {
        "完整性": round(completeness * 100, 2),
        "一致性": round(consistency * 100, 2),
        "准确性": round(accuracy * 100, 2),
        "时效性": round(timeliness * 100, 2),
        "唯一性": round(uniqueness * 100, 2),
    }

scores = quality_score(df)
print("数据质量评分（满分 100）:")
for k, v in scores.items():
    print(f"  {k}: {v}")

# 雷达图
labels = list(scores.keys())
values = list(scores.values()) + [scores["完整性"]]
angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(projection="polar"))
ax.plot(angles, values, "o-", linewidth=2)
ax.fill(angles, values, alpha=0.25)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels)
ax.set_ylim(0, 100); ax.set_title("数据质量 5 维评分", pad=20)
plt.show()

## 总结
本 notebook 完整实现了 DataPilot 应用的全部功能。学员可：
1. 把 work_orders.csv 替换为自己的数据
2. 调整异常检测阈值、KNN k 值等参数
3. 添加更多派生特征
4. 把评分阈值改为公司标准

**与 DataPilot 网页版的等价性**：所有算法逻辑、阈值默认值、可视化形式与浏览器版本一一对应。
